In [1]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [2]:
# --- 1. Universal Styling & Clean Aesthetics ---
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Calibri', 'DejaVu Serif'],
    'font.size': 24,               
    'axes.labelsize': 24,          
    'axes.titlesize': 30,          
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,         # Smaller legend size as requested
    'axes.grid': True,             
    'grid.alpha': 0.3,
    'savefig.dpi': 300,            
    'savefig.bbox': 'tight'        
})

In [3]:
# Lock in colors so methods match across ALL figures natively
method_colors = {
    'RefitRotate': '#7f7f7f', # gray
    'DynamicBVH':  '#ff7f0e', # orange
    'KineticBVH':  '#2ca02c', # green
    'AwareBVH':    '#1f77b4'  # blue
}

In [4]:
# --- 2. Data Loading Logic ---
MESH_RESOLUTIONS = {
    'Suzanne': [('4K', 3936), ('16K', 15744), ('63K', 62976), ('252K', 251904)],
    'Dragon':  [('3K', 3402), ('14K', 13614), ('54K', 54456), ('218K', 217826)],
    'Plane':   [('4K', 4352), ('17K', 16896), ('67K', 66560), ('264K', 264192)],
}

In [5]:
def load_data():
    files = glob.glob("benchall-*.csv")
    summary_list, frames_list = [], []
    for f in files:
        with open(f, 'r') as file:
            header = file.readline().strip('# \n')
            meta = dict(item.split('=') for item in header.split(','))
        df = pd.read_csv(f, comment='#')
        method, deformation, triangles = meta['method'], meta['deformation'], int(meta['triangles'])
        mesh_name = next((m for m in MESH_RESOLUTIONS.keys() if m.lower() in f.lower()), "Unknown")
        
        stats = df.mean(numeric_only=True).to_dict()
        stats.update(meta)
        stats['mesh'], stats['triangles'] = mesh_name, triangles
        summary_list.append(stats)
        
        df_f = df.copy()
        df_f['method'], df_f['deformation'], df_f['mesh'], df_f['triangles'] = method, deformation, mesh_name, triangles
        frames_list.append(df_f)
            
    df_stats = pd.DataFrame(summary_list)
    df_frames = pd.concat(frames_list) if frames_list else pd.DataFrame()
    
    m_order = ['RefitRotate', 'KineticBVH', 'DynamicBVH', 'AwareBVH']
    if not df_stats.empty: df_stats['method'] = pd.Categorical(df_stats['method'], categories=m_order, ordered=True)
    if not df_frames.empty: df_frames['method'] = pd.Categorical(df_frames['method'], categories=m_order, ordered=True)
        
    return df_stats, df_frames

In [6]:
df_stats, df_frames = load_data()

In [7]:
plt.rcParams.update({  
    'axes.labelsize': 30,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
})

In [8]:
# --- 3. Plot Generation (Exporting Individual PDFs) ---

# FIG 1: Scaling Curves (3 Individual PDFs)
for mesh in ['Suzanne', 'Dragon', 'Plane']:
    data = df_stats[(df_stats['mesh'] == mesh) & (df_stats['deformation'] == 'Localized')].sort_values('triangles')
    if not data.empty:
        plt.figure(figsize=(6, 5))
        sns.lineplot(data=data, x='triangles', y='updateMs', hue='method', palette=method_colors, marker='o', linewidth=3, markersize=8)
        # plt.title(f"{mesh}")
        plt.xlabel("Triangles")
        plt.ylabel("Update Time (ms)")
        plt.legend(title="")
        sns.despine()
        plt.tight_layout()
        plt.savefig(f"_new_plots\\fig1_{mesh.lower()}.pdf")
        plt.close()

In [9]:
plt.rcParams.update({     
    'axes.labelsize': 24,   
    'xtick.labelsize': 24,  
    'ytick.labelsize': 16,     
})

In [10]:
# FIG 2: Overhead (Single PDF)
data_loc = df_stats[df_stats['deformation'] == 'Localized'].copy()
if not data_loc.empty:
    max_tris_mask = data_loc.groupby('mesh')['triangles'].transform(max)
    data_high_res = data_loc[data_loc['triangles'] == max_tris_mask]
    plt.figure(figsize=(8, 6))
    
    # Added the order parameter here
    sns.barplot(data=data_high_res, x='mesh', y='updateMs', hue='method', 
                palette=method_colors, order=['Suzanne', 'Dragon', 'Plane'])
    
    # plt.title("BVH Overhead (Highest Resolution)")
    plt.xlabel("")
    plt.ylabel("Update Time (ms)")
    plt.legend(title="")
    sns.despine()
    plt.tight_layout()
    plt.savefig("_new_plots/fig2_overhead.pdf")
    plt.close()

C:\Users\o1a2h\AppData\Local\Temp\ipykernel_25300\1121962696.py:4: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  max_tris_mask = data_loc.groupby('mesh')['triangles'].transform(max)


In [11]:
# FIG 3: Locality Impact (Single PDF)
data_plane = df_stats[df_stats['mesh'] == 'Plane']
if not data_plane.empty:
    max_p = data_plane['triangles'].max()
    data_p_high = data_plane[data_plane['triangles'] == max_p]
    plt.figure(figsize=(8, 6))
    sns.barplot(data=data_p_high, x='deformation', y='updateMs', hue='method', palette=method_colors, order=['SineWave', 'Twist', 'Localized'])
    # plt.title("Deformation Locality Impact")
    plt.xlabel("")
    plt.ylabel("Update Time (ms)")
    plt.legend(title="")
    sns.despine()
    plt.tight_layout()
    plt.savefig("_new_plots\\fig3_locality.pdf")
    plt.close()

In [12]:
plt.rcParams.update({
    'axes.labelsize': 30,   
    'xtick.labelsize': 20,  
    'ytick.labelsize': 20,    
})

In [13]:
# FIG 4: Vertex Work (3 Individual PDFs)
method_styles = {
    'RefitRotate': {'ls': '-',  'lw': 3, 'marker': 'o', 'color': method_colors['RefitRotate']},
    'DynamicBVH':  {'ls': '--', 'lw': 3, 'marker': 's', 'color': method_colors['DynamicBVH']},
    'KineticBVH':  {'ls': ':',  'lw': 3, 'marker': '^', 'color': method_colors['KineticBVH']},
    'AwareBVH':    {'ls': '-',  'lw': 3, 'marker': 'D', 'color': method_colors['AwareBVH']}
}
for mesh in ['Suzanne', 'Dragon', 'Plane']:
    data_m = df_stats[(df_stats['mesh'] == mesh) & (df_stats['deformation'] == 'Localized')].sort_values('triangles')
    if not data_m.empty:
        plt.figure(figsize=(6, 5))
        for m, style in method_styles.items():
            sub = data_m[data_m['method'] == m]
            if not sub.empty:
                plt.plot(sub['triangles'], sub['verticesChecked'], label=m, **style)
        plt.yscale('log')
        # plt.title(f"{mesh}")
        plt.xlabel("Triangles")
        plt.ylabel("Vertices Checked")
        plt.legend(title="", fontsize=12)
        sns.despine()
        plt.tight_layout()
        plt.savefig(f"_new_plots\\fig4_{mesh.lower()}.pdf")
        plt.close()

In [14]:
# FIG 5: Timeline Stability (Single PDF)
if not df_frames.empty:
    max_d = df_frames[df_frames['mesh'] == 'Dragon']['triangles'].max()
    df_f5 = df_frames[(df_frames['mesh'] == 'Dragon') & (df_frames['triangles'] == max_d) & (df_frames['deformation'] == 'Localized')].copy()
    
    if not df_f5.empty:
        # Define interpolation function to normalize every run to exactly 0-100%
        def get_averaged_timeline(df, metric_col):
            common_x = np.linspace(0, 100, 300)
            all_y_interpolated = []
            for run_id in df['run'].unique():
                run_data = df[df['run'] == run_id].sort_values('frame')
                
                # Normalize this specific run's frames to 0-100%
                f_min, f_max = run_data['frame'].min(), run_data['frame'].max()
                x_vals = (run_data['frame'] - f_min) / (f_max - f_min) * 100
                y_vals = run_data[metric_col]
                
                all_y_interpolated.append(np.interp(common_x, x_vals, y_vals))
            return common_x, np.mean(all_y_interpolated, axis=0)

        plt.figure(figsize=(8, 6))
        
        # Plot each method using the normalized interpolation
        for m in df_stats['method'].cat.categories:
            df_m = df_f5[df_f5['method'] == m]
            if not df_m.empty:
                x, y = get_averaged_timeline(df_m, 'totalMs')
                plt.plot(x, y, linestyle='-', linewidth=3, color=method_colors.get(m, 'black'), label=m)
                
        # plt.title("Averaged Temporal Stability (Normalized)")
        plt.xlabel("Simulation Progress (%)")
        plt.ylabel("Total Time (ms)")
        plt.xlim(0, 100)
        plt.legend(title="")
        sns.despine()
        plt.tight_layout()
        plt.savefig("_new_plots/fig5_timeline.pdf")
        plt.close()

In [15]:
plt.rcParams.update({
    'axes.labelsize': 28,   
    'xtick.labelsize': 18,  
    'ytick.labelsize': 18,    
})

In [16]:
# FIG 6: Phase Breakdown from "maybe.ipynb" (4 Individual PDFs)
def get_averaged_line(df, metric_col):
    common_x = np.linspace(0, 100, 300)
    all_y_interpolated = []
    for run_id in df['run'].unique():
        run_data = df[df['run'] == run_id].sort_values('cycleProgress')
        x_vals = run_data['cycleProgress'] * 100
        y_vals = run_data[metric_col]
        all_y_interpolated.append(np.interp(common_x, x_vals, y_vals))
    return common_x, np.mean(all_y_interpolated, axis=0)

if not df_frames.empty:
    max_d = df_frames[df_frames['mesh'] == 'Dragon']['triangles'].max()
    data_maybe = df_frames[(df_frames['mesh'] == 'Dragon') & (df_frames['triangles'] == max_d) & (df_frames['deformation'] == 'Twist')].copy()

    if not data_maybe.empty:
        # Calculate progress natively if cycleProgress doesn't strictly exist
        if 'cycleProgress' not in data_maybe.columns:
            data_maybe['cycleProgress'] = (data_maybe['frame'] - data_maybe['frame'].min()) / (data_maybe['frame'].max() - data_maybe['frame'].min())
            
        metrics = ['deformMs', 'updateMs', 'queryMs', 'totalMs']
        titles = ['Deformation Time', 'BVH Update Time', 'Collision Query Time', 'Total Time']
        
        for i, metric in enumerate(metrics):
            plt.figure(figsize=(6, 5))
            for m in df_stats['method'].cat.categories:
                df_m = data_maybe[data_maybe['method'] == m]
                if not df_m.empty:
                    x, y = get_averaged_line(df_m, metric)
                    plt.plot(x, y, linestyle='-', linewidth=2.5, color=method_colors.get(m, 'black'), label=m)
                    
            # plt.title(titles[i])
            plt.xlabel('Cycle Progress (%)')
            plt.ylabel('Average Time (ms)')
            plt.xlim(0, 100)
            plt.ylim(bottom=0)
            
            # --- LEGEND PLACEMENT LOGIC ---
            if metric not in ['updateMs', 'totalMs']:
                plt.legend(title="")
                
            sns.despine()
            plt.tight_layout()
            plt.savefig(f"_new_plots/fig6_twist_{metric.replace('Ms','')}.pdf")
            plt.close()